# Underthesea NER + Regex Test

This notebook tests `underthesea.ner` as the NER model and then adds the same regex recognizer used in `01_pipeline_demonstration.ipynb`. It follows the same evaluation flow used for the spaCy baseline comparison.

## Setup

In [ ]:
import os
import sys

project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import pandas as pd
from datasets import load_dataset

from src.pipeline.BasePipeline import PIIPipeline
from src.pipeline.Evaluator import PIIEvaluator
from src.pipeline.NERWrappers.BaseNERWrapper import BaseNERWrapper
from src.pipeline.Recognizers.CustomPatternRecognizer import CustomPatternRecognizer
from src.pipeline.Recognizers.DeepLearningRecognizer import DeepLearningRecognizer
from src.pipeline.Recognizers.SpacyRecognizer import SpacyRecognizer
from src.pipeline.Utils import display_evaluation_results, with_prefix, plot_step_progress, plot_per_entity_comparison

from underthesea import ner as underthesea_ner

## Underthesea NER Wrapper

In [ ]:
class UndertheseaNER(BaseNERWrapper):
    """Adapter for underthesea.ner that returns the project NER wrapper format."""

    label_mapping = {
        "PER": "PERSON",
        "LOC": "LOCATION",
        "ORG": "ORGANIZATION",
        "MISC": "MISC",
    }

    def __init__(self, score=0.85):
        self.score = score
        self._is_loaded = False

    def load(self):
        self._is_loaded = True

    def unload(self):
        self._is_loaded = False

    @property
    def is_loaded(self):
        return self._is_loaded

    def _token_offsets(self, text, tagged_tokens):
        offsets = []
        cursor = 0
        for word, *_ in tagged_tokens:
            normalized = word.replace("_", " ")
            start = text.find(normalized, cursor)
            if start == -1:
                start = text.find(normalized)
            if start == -1:
                offsets.append((None, None))
                continue
            end = start + len(normalized)
            offsets.append((start, end))
            cursor = end
        return offsets

    def predict_entities(self, text: str):
        tagged_tokens = underthesea_ner(text)
        offsets = self._token_offsets(text, tagged_tokens)
        entities = []
        active = None

        for token, offset in zip(tagged_tokens, offsets):
            word, _pos, _chunk, ner_tag = token
            start, end = offset
            if start is None or end is None:
                continue

            if ner_tag == "O":
                if active is not None:
                    entities.append(active)
                    active = None
                continue

            bio_tag, raw_label = ner_tag.split("-", 1)
            entity_type = self.label_mapping.get(raw_label, raw_label)

            if bio_tag == "B" or active is None or active["entity_type"] != entity_type:
                if active is not None:
                    entities.append(active)
                active = {
                    "entity_type": entity_type,
                    "start": start,
                    "end": end,
                    "score": self.score,
                    "word": text[start:end],
                }
            else:
                active["end"] = end
                active["word"] = text[active["start"]:end]

        if active is not None:
            entities.append(active)

        return entities

## Smoke Test

In [ ]:
sample_text = "Xin chào, tôi là Nguyễn Văn A. Số điện thoại của tôi là 0912345678. Tôi sống tại Hà Nội và làm việc tại Công ty ABC."

underthesea_wrapper = UndertheseaNER()
underthesea_wrapper.load()
underthesea_wrapper.predict_entities(sample_text)

## Dataset Selection

In [ ]:
from src.pipeline.Utils import load_hf_token
split_choice = "test"
sample_size = 5000

print("Loading dataset: nguyenlamtung/pii-masking-95k-preencoded")
dataset = load_dataset("nguyenlamtung/pii-masking-95k-preencoded", token=load_hf_token())

available_splits = list(dataset.keys())
print("Available splits in dataset:", available_splits)

if split_choice == "all":
    selected_splits = available_splits
else:
    mapped_split = "validation" if split_choice == "val" and "validation" in dataset else split_choice
    selected_splits = [mapped_split] if mapped_split in dataset else ["train"]

dfs = []
for split in selected_splits:
    split_df = dataset[split].to_pandas()
    if len(split_df) > sample_size:
        split_df = split_df.sample(n=sample_size, random_state=42)
    split_df["split"] = split
    dfs.append(split_df)

df_eval = pd.concat(dfs, ignore_index=True)
if "text" in df_eval.columns and "source_text" not in df_eval.columns:
    df_eval["source_text"] = df_eval["text"]

print(f"Loaded and sampled {len(df_eval)} rows across splits: {selected_splits}")

## Evaluate Underthesea

In [ ]:
evaluator = PIIEvaluator()
all_step_results = []
all_per_entity_results = []


def evaluate_pipeline(step_name, pipeline):
    print(f"Starting {step_name} evaluation...")
    pipeline.load_model()
    overall, per_entity = evaluator.evaluate_presidio(
        df_eval=df_eval,
        analyzer=pipeline,
        language="vi",
        use_type_mapping=True,
        return_per_entity=True,
    )

    prefixed = with_prefix("mapped_types", overall)
    prefixed["step_name"] = step_name
    all_step_results.append(prefixed)

    per_entity_df = pd.DataFrame.from_dict(per_entity, orient="index").reset_index().rename(columns={"index": "entity_type"})
    per_entity_df["step_name"] = step_name
    all_per_entity_results.append(per_entity_df)

    print(f"\n{step_name} Metrics:")
    for key, value in overall.items():
        print(f"- {key}: {value}")

    print(f"\n{step_name} Per-Entity Metrics:")
    for entity, metrics in per_entity.items():
        print(f"- {entity}: Precision={metrics['precision']:.4f}, Recall={metrics['recall']:.4f}, F1={metrics['f1']:.4f}")


# Build recognizers
underthesea_recognizer = DeepLearningRecognizer(
    ner_wrapper=UndertheseaNER(),
    supported_entities=["PERSON", "LOCATION", "ORGANIZATION", "MISC"],
    lang_code="vi",
)
custom_patterns = CustomPatternRecognizer()

# Create a lightweight dummy spaCy recognizer + analyzer so Presidio won't try to download default models
class _DummyRegistry:
    def __init__(self):
        self._recognizers = []
    def add_recognizer(self, r):
        self._recognizers.append(r)
    @property
    def recognizers(self):
        return list(self._recognizers)

class DummyAnalyzer:
    def __init__(self):
        self.registry = _DummyRegistry()
    def analyze(self, text, language=None, score_threshold=0.0):
        results = []
        for recognizer in self.registry.recognizers:
            # Pass the recognizer's supported_entities so entity filters inside recognizers still work
            ents = getattr(recognizer, 'supported_entities', None) or getattr(recognizer, 'supported_entity', None) or None
            try:
                res = recognizer.analyze(text=text, entities=ents, nlp_artifacts=None)
            except TypeError:
                # Some recognizers may have a different signature; try positional fallback
                res = recognizer.analyze(text, ents, None)
            if res:
                results.extend(res)
        return results

class DummySpacyRecognizer:
    def __init__(self, lang_code='vi'):
        # Provide lang_code so PIIPipeline.predict can determine default language
        self.lang_code = lang_code
        self.analyzer = DummyAnalyzer()
    def load_model(self):
        return
    def unload_model(self):
        self.analyzer = None

# Step 0: Regex-only (no DL NER)
dummy_spacy = DummySpacyRecognizer()
pipeline_regex_only = PIIPipeline(
    spacy_recognizer=dummy_spacy,
    recognizers=[custom_patterns],
)
evaluate_pipeline("Step 0: Regex Only", pipeline_regex_only)

# Step 1: NER-only (Deep learning recognizer only)
pipeline_underthesea = PIIPipeline(
    spacy_recognizer=dummy_spacy,
    recognizers=[underthesea_recognizer],
)
evaluate_pipeline("Step 1: Underthesea NER", pipeline_underthesea)


## Evaluate Underthesea + Regex

In [ ]:
underthesea_regex_recognizer = DeepLearningRecognizer(
    ner_wrapper=UndertheseaNER(),
    supported_entities=["PERSON", "LOCATION", "ORGANIZATION", "MISC"],
    lang_code="vi",
)
custom_patterns = CustomPatternRecognizer()
pipeline_underthesea_regex = PIIPipeline(
    spacy_recognizer=SpacyRecognizer(model_name="xx_ent_wiki_sm", lang_code="vi"),
    recognizers=[underthesea_regex_recognizer, custom_patterns],
)

evaluate_pipeline("Step 2: Underthesea NER + Regex", pipeline_underthesea_regex)

## Compare Results

In [ ]:
%load_ext autoreload
%autoreload 2

overall_df = pd.DataFrame(all_step_results)
per_entity_df = pd.concat(all_per_entity_results, ignore_index=True) if all_per_entity_results else None

display_evaluation_results(overall_df, per_entity_df)

# Plot overall step progress and per-entity comparisons
plot_step_progress(overall_df)
if per_entity_df is not None:
    steps_order = overall_df['step_name'].tolist()
    plot_per_entity_comparison(per_entity_df, steps_order=steps_order)